### Sample Utils Notebook ###
* Download Clip: Downloads open AI clip model and returns model object
* Load Images: loads images from a given directory. Returns either an array or pillow object
* Clip Predictor: returns dataframe of predictions based on image input and label input

In [ ]:
import pandas as pd
from feature_extraction_utils import load_images, clip_predictor, download_clip

# define classifier models
classifier = download_clip()


**Define Sample Labels for vehicle type and color**


In [ ]:
color_labels = ["white", "black", "red", "orange", "yellow", "green", "blue", "purple"]
vehicle_labels = ["car", "bus", "motorcycle", "license plate"]

**Load sample images from directory**

In [ ]:
image_folder = r"../data/formatted/license_plate_detection/train/images"
image, fp = load_images(directory= image_folder, num_img= 1, img_obj= True)

**Predict label probability based on labels and input image(s)**

In [ ]:
import pandas as pd

vehicle_list = []
val = clip_predictor(image, vehicle_labels, classifier)

val = val.astype(float).round(4)
print(val)

**Show Image and Probabilities for Sample Image**

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
sample_image = Image.open(fp)
plt.imshow(sample_image)

# Define box properties
props = dict(boxstyle='round', facecolor='wheat', alpha=0.5)

# Add text box at (x=1.5, y=6)
w, h = sample_image.size
plt.text(0, h*.15, val, fontsize=12, bbox=props)
plt.xticks([])
plt.yticks([])
plt.show()

In [ ]:
image_folder = r"../data/formatted/license_plate_detection/train/images"
results = load_images(directory= image_folder, num_img= 100, img_obj= True)

vehicle_list = []
for idx, item in enumerate(results):
    image, file_path = item[0], item[1]
    val = clip_predictor(image, vehicle_labels, classifier)
    val = val.astype(float).round(4)

    val["fp"] = file_path

    vehicle_list.append(val)

    pct_complete = idx/len(results)
    if pct_complete*100 % 10 == 0:
        print(f"{pct_complete*100:.2f}% Complete")

vehicle_df = pd.concat(vehicle_list)

vehicle_df.head(5)

**Save Results to CSV file**

In [ ]:
import os
vehicle_df['top_label'] = vehicle_df.drop(columns = ["fp"]).idxmax(axis=1)
vehicle_df.to_csv(f"C:/Users/{os.getlogin()}/Downloads/vehicles_probabilities.csv")

**For large image sets, save predictions to a .csv file**

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np

def plot_df(dataframe, col_label):
    cols = 3
    n = dataframe.shape[0]
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize = (12, rows * 3) )

    axes_flat = axes.flatten()

    for idx, row in dataframe.iterrows():
        axes_flat[idx].imshow(Image.open(row["fp"]))
        axes_flat[idx].set_xticks([])
        axes_flat[idx].set_yticks([])
        axes_flat[idx].set_title(f'Probability {row[col_label]:.2f}%')

    plt.show()

df_license_plate_only = vehicle_df[vehicle_df["top_label"] == "license plate"].reset_index()
plot_df(df_license_plate_only, "license plate")

In [ ]:
df_motorcycle = vehicle_df[vehicle_df["top_label"] == "motorcycle"].reset_index()

plot_df(df_motorcycle, "motorcycle")